# 1. Imports y carga de datos

In [13]:
import pandas as pd
import unicodedata
import re

df = pd.read_csv('../data/processed/reviews_with_sentimiento.csv')

print(f'shape inicial: {df.shape}')
print(df.columns.to_list())


shape inicial: (3423, 20)
['text', 'rating', 'publishedAtDate', 'restaurante', 'rating_comida', 'rating_servicio', 'rating_ambiente', 'tiempo_espera_reportado', 'likesCount', 'reviewerNumberOfReviews', 'isLocalGuide', 'fecha', 'Year', 'tiene_texto', 'review_texto', 'idioma_detectado', 'sentimiento', 'prob_positivo', 'prob_negativo', 'prob_neutral']


# 2. Preparar el texto para búsqueda de temas

Aquí si queremos texto sin tildes y en minúscula, porque vamos a hacer matching de palabras clave(a diferencia del sentimiento, donde dejamos el texto neutral para pystentimiento)

In [14]:
def normalizar_para_busqueda(texto):
    if pd.isna(texto):
        return ''
    
    texto = texto.lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii','ignore').decode('utf-8')
    return texto


df['texto_normalizado'] = df['review_texto'].apply(normalizar_para_busqueda)

# 3. Definir el diccionario de temas (ajustados a hallazgos reales)

In [15]:
temas = {
    "sabor_comida": ["sazon", "sabor", "rico", "delicioso", "sabroso", "insipido", "soso",
                      "exquisito", "delicia", "buenisima", "buenisimo", "manjar"],
    
    "carne": ["carne", "termino", "punto", "jugosa", "dura", "asado", "churrasco",
              "chuleta", "costilla", "cerdo", "pollo", "bistec", "parrillada",
              "carbon", "brasa", "correosa", "seca"],
    
    "moro": ["moro", "arroz", "menestra", "frejol", "frijol", "lenteja", "guarnicion"],
    
    "atencion_servicio": ["atencion", "servicio", "mesero", "mesera", "amable", "grosero",
                          "despota", "pesimo", "mala atencion", "buena atencion",
                          "personal", "staff", "trato", "amabilidad"],
    
    "tiempo_espera": ["espera", "demora", "rapido", "lento", "tardaron", "minutos",
                      "toco esperar", "tardanza", "demoro", "demoraron", "hora"],
    
    "porciones": ["porcion", "cantidad", "poco", "abundante", "tamano",
                  "pequeño", "grande", "suficiente", "escaso", "generoso", "generosa"],
    
    "precio": ["precio", "caro", "barato", "vale la pena", "costoso",
               "economico", "accesible", "cuenta", "pagar", "dolares", "valor",
               "calidad precio", "costo"],
    
    "ambiente_instalaciones": ["ambiente", "espacio", "aire acondicionado", "mesa",
                               "bano", "musica", "ruido", "television", "local",
                               "decoracion", "limpieza", "limpio", "sucio", "iluminacion"]
}

# 4. Función para detectar temas mencionados por reseña

In [16]:
def detectar_temas(texto_normalizado):
    temas_encontrados = []
    for tema, palabras_clave in temas.items():
        for palabra in palabras_clave:
            patron = r'\b' + re.escape(palabra) + r'\b'
            if re.search(patron, texto_normalizado):
                temas_encontrados.append(tema)
                break
    return temas_encontrados


df['temas_detectados'] = df['texto_normalizado'].apply(detectar_temas)

#Cuántos temas en promedio menciona cada reseña
df['num_temas'] = df['temas_detectados'].apply(len)
print(df['num_temas'].value_counts().sort_index())
 


num_temas
0    1149
1    1287
2     626
3     217
4      93
5      30
6      14
7       7
Name: count, dtype: int64


# 5. Convertir a formato 'long'

In [17]:
df_temas = df.explode('temas_detectados')

df_temas = df_temas[df_temas['temas_detectados'].notna()] # quitamos las que no detectaron ningún tema

print(f'Reseñas originales {len(df)}')
print(f'Filas despúes de explode (reseña x tema): {len(df_temas)}')

Reseñas originales 3423
Filas despúes de explode (reseña x tema): 3845


# 6. Tabal clave % de menciones por tema, por restaurantes

In [18]:
df_temas = df_temas.reset_index(drop=True)
tabla_temas = pd.crosstab(df_temas['restaurante'], df_temas['temas_detectados'], normalize='index') * 100
tabla_temas = tabla_temas.round(1)
display(tabla_temas)

temas_detectados,ambiente_instalaciones,atencion_servicio,carne,moro,porciones,precio,sabor_comida,tiempo_espera
restaurante,,,,,,,,
Casa Res | Steak House (Mall del Sol),12.9,35.3,15.8,2.1,3.9,7.7,15.4,6.8
Don Parrilla Steak House - Urdesa,14.9,31.4,12.4,8.8,8.2,4.6,12.9,6.7
La Casa del Tomahawk,13.8,38.4,8.9,3.2,6.1,7.8,12.3,9.4
La Parrilla Del Ñato - Urdesa,11.7,42.5,17.3,1.1,4.3,6.0,15.2,1.9
MoroGrill - C.C. Las Terrazas,12.9,29.8,7.2,10.7,6.0,10.7,18.8,4.1
Parrillada Punta Del Este,15.4,39.5,14.2,1.9,2.5,5.6,17.9,3.1
Parrillada Restaurant El Dorado Sauces 3,14.4,37.0,11.6,2.8,6.9,6.9,17.6,2.8
Rukito Grill&Drink - Alborada,11.8,31.0,8.2,9.1,7.7,7.1,18.0,7.1


# 7. Sentimiento promedio por tema, por restaurante

In [19]:
# Convertimos sentimiento a score numérico para poder promediar
mapa_sentimiento  = {'POS': 1, 'NEU':0,'NEG':-1}
df_temas['sentimiento_score'] = df_temas['sentimiento'].map(mapa_sentimiento)

tabla_sentimiento_tema = df_temas.groupby(['restaurante','temas_detectados'])['sentimiento_score'].mean().unstack()
tabla_sentimiento_tema = tabla_sentimiento_tema.round(2)
display(tabla_sentimiento_tema)

temas_detectados,ambiente_instalaciones,atencion_servicio,carne,moro,porciones,precio,sabor_comida,tiempo_espera
restaurante,,,,,,,,
Casa Res | Steak House (Mall del Sol),0.46,0.55,0.44,0.27,0.10,0.30,0.54,-0.37
Don Parrilla Steak House - Urdesa,0.48,0.48,0.08,0.00,0.06,0.00,0.48,-0.38
La Casa del Tomahawk,0.17,0.26,-0.29,-0.22,-0.12,-0.14,0.28,-0.61
La Parrilla Del Ñato - Urdesa,0.40,0.70,0.44,1.00,0.06,0.23,0.66,0.57
MoroGrill - C.C. Las Terrazas,0.76,0.64,0.43,0.47,0.68,0.47,0.63,-0.15
Parrillada Punta Del Este,0.56,0.77,0.83,1.00,0.50,0.11,0.69,0.20
Parrillada Restaurant El Dorado Sauces 3,0.68,0.68,0.56,0.83,0.27,0.73,0.74,0.17
Rukito Grill&Drink - Alborada,0.39,0.56,0.14,0.41,0.23,0.28,0.56,-0.02


In [20]:
#Conteo de reseñas por tema y restaurante
tabla_conteo = df_temas.groupby(['restaurante','temas_detectados']).size().unstack(fill_value=0)
display(tabla_conteo)

temas_detectados,ambiente_instalaciones,atencion_servicio,carne,moro,porciones,precio,sabor_comida,tiempo_espera
restaurante,,,,,,,,
Casa Res | Steak House (Mall del Sol),67,183,82,11,20,40,80,35
Don Parrilla Steak House - Urdesa,29,61,24,17,16,9,25,13
La Casa del Tomahawk,116,322,75,27,51,65,103,79
La Parrilla Del Ñato - Urdesa,43,157,64,4,16,22,56,7
MoroGrill - C.C. Las Terrazas,41,95,23,34,19,34,60,13
Parrillada Punta Del Este,25,64,23,3,4,9,29,5
Parrillada Restaurant El Dorado Sauces 3,31,80,25,6,15,15,38,6
Rukito Grill&Drink - Alborada,145,381,101,112,95,87,221,87


# 8. Complementar con los datos estructurados de Google (Comida/Servicio/Ambiente)

In [21]:
ratings_estructurados = df.groupby('restaurante')[['rating_comida','rating_servicio','rating_ambiente']].mean().round(2)
display(ratings_estructurados)

,rating_comida,rating_servicio,rating_ambiente
restaurante,,,
Casa Res | Steak House (Mall del Sol),3.80,3.80,4.15
Don Parrilla Steak House - Urdesa,4.04,4.11,4.19
La Casa del Tomahawk,3.98,3.69,4.06
La Parrilla Del Ñato - Urdesa,4.35,4.54,4.26
MoroGrill - C.C. Las Terrazas,4.62,4.67,4.72
Parrillada Punta Del Este,4.57,4.40,4.19
Parrillada Restaurant El Dorado Sauces 3,4.56,4.47,4.43
Rukito Grill&Drink - Alborada,4.46,4.20,4.24


# 9. Tiempo de espera reportado (dato estructurado directo)

In [22]:
display(df.groupby('restaurante')['tiempo_espera_reportado'].value_counts(normalize=True).unstack().round(2) * 100)

tiempo_espera_reportado,De 10 a 30 min,De 30 a 60 min,Hasta 10 min,Más de 1 hora,Sin espera
restaurante,,,,,
Casa Res | Steak House (Mall del Sol),25.0,17.0,8.0,NaN,50.0
Don Parrilla Steak House - Urdesa,NaN,NaN,67.0,NaN,33.0
La Casa del Tomahawk,18.0,9.0,28.0,2.0,43.0
La Parrilla Del Ñato - Urdesa,NaN,NaN,17.0,NaN,83.0
MoroGrill - C.C. Las Terrazas,20.0,NaN,30.0,NaN,50.0
Parrillada Punta Del Este,33.0,NaN,67.0,NaN,NaN
Parrillada Restaurant El Dorado Sauces 3,8.0,8.0,17.0,NaN,67.0
Rukito Grill&Drink - Alborada,46.0,7.0,23.0,7.0,18.0


# 10. Exportar resultados

In [23]:
df.to_csv('../data/processed/reviews_with_topics.csv',index=False)
df_temas.to_csv('../data/final/reviews_topics_long.csv',index=False)
tabla_sentimiento_tema.to_csv('../data/final/sentiment_by_topic_restaurnat.csv')
ratings_estructurados.to_csv('../data/final/structured_ratings_by_restaurant.csv')

print('Archivos finales guardados en data/final/')

Archivos finales guardados en data/final/
